# 02 — SQLAlchemy Data Layer Demo (Document Metadata)

This notebook demonstrates the SQLAlchemy ORM patterns from chapters 02 and 05
(`02-building-the-service-with-fastapi.md`, `05-secrets-management-with-key-vault-and-sql-integration.md`)
using an **in-memory SQLite** database via SQLAlchemy — no external database server, no connection
string, no Azure SQL required. The declarative model and query *shapes* (indexes, soft-delete, audit
columns, tenant scoping, cursor pagination) are the same ideas that run against Azure SQL in production;
only the underlying SQL dialect differs slightly (SQLite instead of T-SQL) — SQLAlchemy itself is what
made the code identical in shape regardless of which dialect sits underneath.

**What this demonstrates:**
1. A declarative `Document` model matching chapter 05's schema (metadata, audit columns, soft-delete,
   `tenant_id`), with indexes
2. Why parameterized queries are **automatic** with SQLAlchemy's query-building API — no separate
   discipline required, unlike hand-written cursor calls
3. Soft-delete expressed as an ORM attribute update, not a raw `UPDATE` statement
4. A **tenant-scoped repository class** that enforces the `tenant_id` filter structurally (chapter 05's
   "one chokepoint" pattern), demonstrated with two tenants (`hsbc`, `bofa`) sharing one database
5. Cursor-style pagination expressed through SQLAlchemy's `.where()`/`.order_by()`/`.limit()`
6. Verifying the index is actually used, via `EXPLAIN QUERY PLAN`


In [1]:
from datetime import datetime, timezone
from typing import Optional

from sqlalchemy import (
    create_engine, select, text,
    String, Integer, BigInteger, Boolean, DateTime, Index,
)
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, sessionmaker, Session
from sqlalchemy.pool import StaticPool

print("SQLAlchemy ORM demo — in-memory SQLite, no external database required.")


SQLAlchemy ORM demo — in-memory SQLite, no external database required.


## 1. The declarative `Document` model, matching chapter 05's schema

Metadata columns, a `status` state machine, audit columns (`created_by`, `created_at`, `updated_at`),
soft-delete columns (`is_deleted`, `deleted_at`, `deleted_by`), and — because this service ran in
production for **two** banking clients on the same database (chapter 05) — a `tenant_id` column that
leads the composite index, exactly as in the real schema.


In [2]:
class Base(DeclarativeBase):
    pass


class Document(Base):
    __tablename__ = "documents"
    __table_args__ = (
        # tenant_id leads every composite index — every real query filters by tenant first (chapter 05)
        Index("idx_documents_tenant_status_created", "tenant_id", "status", "created_at"),
        Index("idx_documents_tenant_created_by", "tenant_id", "created_by"),
    )

    # NOTE: Azure SQL's actual column type is BIGINT (chapter 05); this demo uses plain Integer for the
    # primary key because SQLite only wires up autoincrement for its INTEGER PRIMARY KEY rowid alias.
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    tenant_id: Mapped[str] = mapped_column(String(32), nullable=False)   # 'hsbc' | 'bofa'
    filename: Mapped[str] = mapped_column(String(260), nullable=False)
    blob_path: Mapped[str] = mapped_column(String(1024), nullable=False)
    content_type: Mapped[str] = mapped_column(String(128), nullable=False)
    size_bytes: Mapped[int] = mapped_column(BigInteger, nullable=False)
    status: Mapped[str] = mapped_column(String(32), nullable=False, default="uploaded")

    created_by: Mapped[str] = mapped_column(String(128), nullable=False)
    created_at: Mapped[datetime] = mapped_column(DateTime, default=lambda: datetime.now(timezone.utc))
    updated_at: Mapped[datetime] = mapped_column(DateTime, default=lambda: datetime.now(timezone.utc),
                                                  onupdate=lambda: datetime.now(timezone.utc))

    is_deleted: Mapped[bool] = mapped_column(Boolean, nullable=False, default=False)
    deleted_at: Mapped[Optional[datetime]] = mapped_column(DateTime, nullable=True)
    deleted_by: Mapped[Optional[str]] = mapped_column(String(128), nullable=True)


engine = create_engine(
    "sqlite:///:memory:",
    connect_args={"check_same_thread": False},
    poolclass=StaticPool,
)
SessionLocal = sessionmaker(bind=engine, autoflush=False, expire_on_commit=False)
Base.metadata.create_all(engine)

with engine.connect() as conn:
    rows = conn.execute(text("SELECT name, tbl_name FROM sqlite_master WHERE type = 'index'")).fetchall()
print("Schema + indexes created via SQLAlchemy's declarative Base.metadata.create_all():")
for r in rows:
    print(" - index", repr(r[0]), "on", repr(r[1]))


Schema + indexes created via SQLAlchemy's declarative Base.metadata.create_all():
 - index 'idx_documents_tenant_status_created' on 'documents'
 - index 'idx_documents_tenant_created_by' on 'documents'


## 2. Inserting rows through the ORM — parameterized by construction

There's no `cursor.execute(..., params)` call to remember here — `session.add(Document(...))` builds an
`INSERT` with every value bound as a parameter automatically, because that's simply how SQLAlchemy's
Core/ORM layer generates SQL. This is the "reduced injection surface" argument from chapters 02 and 05
made concrete: the unsafe string-formatting pattern isn't just discouraged, it's not part of this API
at all.

Two tenants — `hsbc` and `bofa` — insert documents into the **same table**, exactly like the real
service's shared-database, `tenant_id`-enforced model (chapter 05).


In [3]:
db: Session = SessionLocal()

hsbc_doc1 = Document(tenant_id="hsbc", filename="hsbc-q3-compliance-policy.pdf",
                      blob_path="hsbc/2026/07/hsbc-q3-compliance-policy.pdf",
                      content_type="application/pdf", size_bytes=245_612,
                      created_by="abhishek.kumar@capco.com")
hsbc_doc2 = Document(tenant_id="hsbc", filename="hsbc-onboarding-guide.docx",
                      blob_path="hsbc/2026/07/hsbc-onboarding-guide.docx",
                      content_type="application/vnd.openxmlformats-officedocument.wordprocessingml.document",
                      size_bytes=88_120, created_by="priya.sharma@capco.com")
bofa_doc1 = Document(tenant_id="bofa", filename="bofa-incident-runbook-v2.pdf",
                      blob_path="bofa/2026/07/bofa-incident-runbook-v2.pdf",
                      content_type="application/pdf", size_bytes=512_004,
                      created_by="abhishek.kumar@capco.com")

db.add_all([hsbc_doc1, hsbc_doc2, bofa_doc1])
db.commit()
for d in (hsbc_doc1, hsbc_doc2, bofa_doc1):
    db.refresh(d)

print("Inserted document ids:", hsbc_doc1.id, hsbc_doc2.id, bofa_doc1.id)
print("Tenants represented:", {d.tenant_id for d in (hsbc_doc1, hsbc_doc2, bofa_doc1)})


Inserted document ids: 1 2 3
Tenants represented: {'bofa', 'hsbc'}


## 3. Why parameterized queries matter — a concrete SQL injection comparison

This cell shows the vulnerable pattern (raw string formatting, the kind of code an ORM's query API
makes unnecessary) side by side with the safe pattern (SQLAlchemy's `select(...)`), using a
deliberately malicious-looking filter value. **Only the safe version is ever used against the real
table** — the vulnerable version below is shown as a string that would be sent to the database, never
actually executed.


In [4]:
# A value an attacker might supply if the API naively trusted user input in a filter
malicious_value = "abhishek.kumar@capco.com' OR '1'='1"

# --- Vulnerable pattern (never do this, ORM or not) ---
unsafe_query = f"SELECT id, filename FROM documents WHERE created_by = '{malicious_value}'"
print("Unsafe query string that would be sent to the database (never executed here):")
print(" ", unsafe_query)
print("  -> the OR '1'='1' clause would make this match every row across every tenant, not just one user's documents.")

# --- Safe pattern: SQLAlchemy's query-building API parameterizes by construction ---
safe_stmt = select(Document.id, Document.filename).where(Document.created_by == malicious_value)
safe_rows = db.execute(safe_stmt).all()
print("\nSafe SQLAlchemy query result (malicious_value treated as literal bound data):", safe_rows)
print("Zero rows matched, because no user actually has that string as their email —")
print("the database never interpreted any part of it as SQL syntax, and there was no string-concatenation")
print("step in the code for a malicious value to hide in.")


Unsafe query string that would be sent to the database (never executed here):
  SELECT id, filename FROM documents WHERE created_by = 'abhishek.kumar@capco.com' OR '1'='1'
  -> the OR '1'='1' clause would make this match every row across every tenant, not just one user's documents.

Safe SQLAlchemy query result (malicious_value treated as literal bound data): []
Zero rows matched, because no user actually has that string as their email —
the database never interpreted any part of it as SQL syntax, and there was no string-concatenation
step in the code for a malicious value to hide in.


## 4. Soft-delete — an ORM attribute update, not a raw `UPDATE` statement

`DELETE /v1/documents/{id}` in the real API (chapter 01, 02) never issues a SQL `DELETE`. Through the
ORM, "soft-delete" is just setting attributes on a loaded object and committing — SQLAlchemy's unit of
work generates the `UPDATE` automatically, still fully parameterized, preserving the row for audit
purposes (chapter 05's compliance rationale).


In [5]:
def soft_delete(session: Session, doc: Document, deleted_by: str) -> None:
    doc.is_deleted = True
    doc.deleted_by = deleted_by
    doc.deleted_at = datetime.now(timezone.utc)
    session.commit()


soft_delete(db, hsbc_doc2, deleted_by="priya.sharma@capco.com")

active_hsbc = db.scalars(
    select(Document).where(Document.tenant_id == "hsbc", Document.is_deleted == False)  # noqa: E712
).all()
print("Active HSBC documents after soft-deleting hsbc_doc2:")
for d in active_hsbc:
    print(" -", d.id, d.filename)

all_hsbc = db.scalars(select(Document).where(Document.tenant_id == "hsbc")).all()
print("\nAll HSBC rows still physically present (audit trail preserved):")
for d in all_hsbc:
    print(" -", d.id, d.filename, "deleted:", d.is_deleted, "at:", d.deleted_at)


Active HSBC documents after soft-deleting hsbc_doc2:
 - 1 hsbc-q3-compliance-policy.pdf

All HSBC rows still physically present (audit trail preserved):
 - 1 hsbc-q3-compliance-policy.pdf deleted: False at: None
 - 2 hsbc-onboarding-guide.docx deleted: True at: 2026-07-15 15:39:40.779190+00:00


## 5. A tenant-scoped repository — the structural enforcement from chapter 05

This is the exact `TenantScopedRepository` pattern from chapter 05: every query built through
`self.query()` is automatically scoped to the caller's tenant, so a missing `tenant_id` filter becomes
structurally impossible in code built on top of this base class, rather than something every call site
has to remember.


In [6]:
class TenantScopedRepository:
    model = Document

    def __init__(self, session: Session, tenant_id: str):
        self.db = session
        self.tenant_id = tenant_id

    def query(self):
        return select(self.model).where(
            self.model.tenant_id == self.tenant_id,
            self.model.is_deleted == False,  # noqa: E712
        )


class DocumentRepository(TenantScopedRepository):
    model = Document

    def get(self, doc_id: int) -> Optional[Document]:
        return self.db.scalar(self.query().where(Document.id == doc_id))

    def list_page(self, status: Optional[str] = None,
                  cursor_created_at: Optional[datetime] = None,
                  page_size: int = 20) -> list[Document]:
        stmt = self.query()
        if status:
            stmt = stmt.where(Document.status == status)
        if cursor_created_at:
            stmt = stmt.where(Document.created_at < cursor_created_at)
        stmt = stmt.order_by(Document.created_at.desc()).limit(page_size)
        return list(self.db.scalars(stmt).all())


hsbc_repo = DocumentRepository(db, tenant_id="hsbc")
bofa_repo = DocumentRepository(db, tenant_id="bofa")

print("HSBC repository sees:", [d.filename for d in hsbc_repo.list_page()])
print("BofA repository sees:", [d.filename for d in bofa_repo.list_page()])

# The isolation guarantee, made concrete: asking the HSBC repository for BofA's document id returns nothing.
cross_tenant_attempt = hsbc_repo.get(bofa_doc1.id)
print("\nHSBC repository looking up BofA's document id ->", cross_tenant_attempt,
      "(None: a query scoped to tenant A never returns a row belonging to tenant B)")


HSBC repository sees: ['hsbc-q3-compliance-policy.pdf']
BofA repository sees: ['bofa-incident-runbook-v2.pdf']

HSBC repository looking up BofA's document id -> None (None: a query scoped to tenant A never returns a row belonging to tenant B)


## 6. Cursor-style pagination through the repository

As described in chapter 05, `OFFSET`-based pagination gets slower with page depth because the database
has to scan and discard every preceding row. Cursor pagination instead carries "the last row I saw" as
an opaque marker (`created_at` here) and asks for rows strictly before it — expressed here purely
through `DocumentRepository.list_page`'s `.where()`/`.order_by()`/`.limit()` chain, an efficient index
seek at any depth using the `idx_documents_tenant_status_created` index created above.


In [7]:
# Insert a couple more HSBC documents so pagination has something to show
extra1 = Document(tenant_id="hsbc", filename="hsbc-audit-report-jan.pdf",
                   blob_path="hsbc/2026/07/hsbc-audit-report-jan.pdf", content_type="application/pdf",
                   size_bytes=90_210, created_by="priya.sharma@capco.com")
extra2 = Document(tenant_id="hsbc", filename="hsbc-audit-report-feb.pdf",
                   blob_path="hsbc/2026/07/hsbc-audit-report-feb.pdf", content_type="application/pdf",
                   size_bytes=91_004, created_by="priya.sharma@capco.com")
db.add_all([extra1, extra2])
db.commit()

page1 = hsbc_repo.list_page(page_size=2)
print("Page 1:")
for d in page1:
    print(" -", d.id, d.filename, d.created_at)

if page1:
    cursor = page1[-1].created_at
    page2 = hsbc_repo.list_page(page_size=2, cursor_created_at=cursor)
    print("\nPage 2 (using cursor from end of page 1):")
    for d in page2:
        print(" -", d.id, d.filename, d.created_at)
    if not page2:
        print(" (no more active HSBC documents)")


Page 1:
 - 5 hsbc-audit-report-feb.pdf 2026-07-15 15:39:40.799490+00:00
 - 4 hsbc-audit-report-jan.pdf 2026-07-15 15:39:40.799485+00:00

Page 2 (using cursor from end of page 1):
 - 1 hsbc-q3-compliance-policy.pdf 2026-07-15 15:39:40.762688


## 7. Inspecting the query plan (bonus: proving the index is used)

SQLite's `EXPLAIN QUERY PLAN` confirms the tenant+status+date filter is served by the composite index
created above, rather than a full table scan — the same kind of check you'd run with T-SQL's execution
plan tooling against Azure SQL before trusting an index in production. This is introspection against the
raw connection (via `text()`), not a "raw SQL" data-access path — the application's actual queries all
go through the ORM/repository above.


In [8]:
with engine.connect() as conn:
    plan = conn.execute(text("""
        EXPLAIN QUERY PLAN
        SELECT id, filename FROM documents
        WHERE tenant_id = 'hsbc' AND is_deleted = 0 AND status = 'uploaded'
        ORDER BY created_at DESC
    """)).fetchall()

for row in plan:
    print(dict(row._mapping))


{'id': 4, 'parent': 0, 'notused': 60, 'detail': 'SEARCH documents USING INDEX idx_documents_tenant_status_created (tenant_id=? AND status=?)'}


## Summary

| Concept from chapter 05 | Demonstrated as |
|---|---|
| Declarative `Document` model (metadata + audit + soft-delete + `tenant_id` columns) | Section 1 |
| Parameterized queries — automatic with the ORM, no separate discipline required | Section 3 |
| Soft-delete as an ORM attribute update | Section 4 |
| Tenant-scoped repository enforcing `tenant_id` structurally (chapter 05's "one chokepoint") | Section 5 |
| Cursor-based pagination via `.where()`/`.order_by()`/`.limit()` | Section 6 |
| Index usage verification | Section 7 |

In production this table lives in Azure SQL, accessed through the same `DocumentRepository` shown here,
with the `Session` bound to a real Azure SQL connection (its connection string resolved from Key Vault
via managed identity, chapter 05) instead of SQLite's in-memory mode — the ORM model, repository
pattern, and query discipline shown here carry over directly; only the connection string and SQL
dialect change.
